[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xylem-lab/GEO-ANOM-dashboard/blob/main/notebooks/06_poultry_corn_loop.ipynb)

# 06 — The poultry → corn loop (first Nutrient Balance Map)

**Question.** Where in Maryland is poultry litter produced close to the corn that could use it, and where is there more litter than nearby corn (candidates for translocation of nutrients or a waste-to-resource facility)? This is the loop from the 10/7 workshop, and the LOI's Task 2 (*Nutrient Demand Mapping → Nutrient Balance Map*) in its first form: poultry supply only, corn demand only.

**How.**
1. **Supply:** every poultry house mapped in Task 1 (notebook 05), as roof area (m²) per 5 km cell. Litter scales with house floor area, and this counts houses whether or not they're in the permit registry.
2. **Demand:** corn acres per cell from the USDA Cropland Data Layer, 2023 (same year as the aerial photos).
3. **Balance index:** m² of poultry house per acre of corn, in each cell and within a hauling distance (default 25 km).
4. **Robustness:** the same index with corn averaged over 2021–2024 (rotation), with soybeans and small grains added, and at 10 / 25 / 50 km.
5. **County cross-check:** birds per m² implied by the AWTF 2017 inventory and our roof area. Compare it with the Extension stocking number.

**Why no N or P numbers yet.** If litter per m² of house (*a*) and crop need per acre (*b*) are constants, surplus means roof m² / acres > *b* / *a*. So the ranking of places in step 3 is already the ranking the N or P map will give. The coefficients (Ginny Rhodes, John Moyle) only decide where the surplus line falls. `B.nutrient_balance()` refuses to run until every coefficient is filled in (section 7).

**Inputs.** `Task1_results/county_*/` from notebook 05 (only finished counties are used); CDL Maryland GeoTIFFs from USDA NASS (~90 MB per year, cached in `Task1_results/reference/`).
**Outputs.** `Task1_results/task2_loop/`: the grid (GeoJSON, opens in Google Earth/QGIS), the county table, the maps, `run_meta.json`.
**Runtime.** A few minutes (mostly the CDL download the first time). No aerial photos are read; this notebook doesn't touch the detector.

In [ ]:
# ── Setup: the same cell works in Google Colab and locally ──────────────────────
import os, sys, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "main"   # branch, tag or commit to run; the exact commit is recorded in run_meta.json

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                     # results go to the shared drive's Task1_results
    ROOT = Path("/content/GEO-ANOM-dashboard")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/xylem-lab/GEO-ANOM-dashboard", str(ROOT)], check=True)
    # always bring the copy up to date (a reused Colab session keeps an old copy)
    head = lambda: subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout
    before = head()
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=True)
    remote = f"origin/{CODE_VERSION}"
    target = remote if subprocess.run(["git", "-C", str(ROOT), "rev-parse", "-q", "--verify", remote],
                                      capture_output=True).returncode == 0 else CODE_VERSION
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", target], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
    if head() != before and "geo_anom" in sys.modules:   # new code, but the old version is already loaded
        raise RuntimeError("Code updated. Runtime -> Restart session, then Run all again.")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
# reload edited geo_anom/ code automatically, without restarting the kernel
if "google.colab" not in sys.modules:   # Colab's IPython can't load autoreload on Python 3.12+ (no `imp`)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

from geo_anom.task1 import env, tiles, region, viz
from geo_anom.task1 import county as C
from geo_anom.task1.kmz import write_kmz
from geo_anom.task1.pipeline import run_pipeline
from geo_anom.task1.evaluate import evaluate_run, load_ground_truth
from geo_anom.task1.pipeline import git_commit
from geo_anom.task2 import cdl, balance as B
print("running in", "Colab" if IN_COLAB else "local", "| results ->", env.results_root())

## Settings

In [ ]:
CELL_KM = 5               # grid cell size
HAUL_KM = 25              # main hauling distance for the "within reach" balance
RADII_KM = [10, 25, 50]   # sensitivity
YEAR = 2023               # CDL year matching the NAIP photos
ROTATION_YEARS = [2021, 2022, 2023, 2024]   # corn averaged over these, for the rotation check
COUNTIES = [n for n in C.all_counties_in_order() if C.is_complete(n)]   # finished in notebook 05
OUT = env.results_root() / "task2_loop"
REF = env.results_root() / "reference"   # CDL copies, so later sessions don't depend on the NASS server
CELL_M = CELL_KM * 1000
print(len(COUNTIES), "finished counties:", ", ".join(COUNTIES))

## 1. Supply side: poultry houses from Task 1

Houses whose centre is inside each finished county, with their roof area measured in UTM metres.

In [ ]:
parts, summaries = [], {}
for n in COUNTIES:
    _, _, bld = C.load_county_run(C.county_out(n))
    bb = C.county_buildings(bld, region.county_boundary(n))
    bb["county"] = n
    parts.append(bb[["county", "geometry"]])
    summaries[n] = json.loads((C.county_out(n) / "county_summary.json").read_text())
houses_gdf = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=region.UTM)
houses_gdf["roof_m2"] = houses_gdf.geometry.area
by_county = houses_gdf.groupby("county").agg(houses=("roof_m2", "size"), roof_m2=("roof_m2", "sum"),
                                             median_house_m2=("roof_m2", "median"))
display(by_county.round(0))
print(f"{len(houses_gdf):,} houses, {houses_gdf.roof_m2.sum() / 1e6:.2f} km² of roof")

## 2. Demand side: corn from the Cropland Data Layer

Corn includes the double-crop classes with corn in them (wheat/corn, oats/corn, barley/corn, corn/soybeans); see `cdl.CROP_GROUPS`. Check: the Maryland total should be near NASS's planted-area survey for the year (2023: ~510,000 acres).

In [ ]:
cdl_info = {y: cdl.ensure_cdl(y, drive_dir=REF) for y in sorted(set([YEAR] + ROTATION_YEARS))}
acres = cdl.class_acres(cdl_info[YEAR]["path"])
group_acres = {g: sum(acres.get(k, 0) for k in codes) for g, codes in cdl.CROP_GROUPS.items()}
display(pd.Series(group_acres, name=f"Maryland acres, CDL {YEAR}").round(0).to_frame())

crops = {"corn": cdl.crop_acres_on_grid(cdl_info[YEAR]["path"], cdl.CROP_GROUPS["corn"], CELL_M),
         "grain": cdl.crop_acres_on_grid(cdl_info[YEAR]["path"],
                                         cdl.CROP_GROUPS["corn"] + cdl.CROP_GROUPS["soybeans_small_grains"], CELL_M)}
rot = [cdl.crop_acres_on_grid(cdl_info[y]["path"], cdl.CROP_GROUPS["corn"], CELL_M) for y in ROTATION_YEARS]
keys = set().union(*[set(d) for d in rot])
crops["corn_rotation_mean"] = {k: float(np.mean([d.get(k, 0.0) for d in rot])) for k in keys}
print({y: round(sum(cdl.crop_acres_on_grid(cdl_info[y]['path'], [1], 1e6).values())) for y in ROTATION_YEARS},
      "acres of class-1 corn per year")

## 3. The grid and the balance index

Each cell: houses, roof m², corn acres; the same sums within each radius; and the index = roof m² per corn acre. Cells are 5 km squares on a fixed UTM grid. The **within-hauling-distance** index (`HAUL_KM`) is the one that matters for hauling: a cell full of houses next to a lot of corn is not a surplus.

The map covers the finished counties only. Corn in counties that aren't finished, and in Delaware / Virginia, isn't in the supply-and-demand comparison.

In [ ]:
h = B.houses_on_grid(houses_gdf, CELL_M)
grid = B.build_grid(h, crops, CELL_M)
for r in RADII_KM:
    grid = B.add_within(grid, ["roof_m2", "corn_acres", "grain_acres", "corn_rotation_mean_acres"], r * 1000, CELL_M)
tag = f"{HAUL_KM}km"
grid["index_cell"] = B.roof_per_acre(grid.roof_m2, grid.corn_acres)
grid["index_haul"] = B.roof_per_acre(grid[f"roof_m2_{tag}"], grid[f"corn_acres_{tag}"])
# which county each cell's centre is in (cells over water or out of state: none)
cty = region.md_counties().to_crs(region.UTM)
cty["county"] = np.where(cty["NAME"].duplicated(keep=False), cty["NAMELSAD"], cty["NAME"])
cent = gpd.GeoDataFrame(grid[["cell_id"]], geometry=grid.centroid, crs=region.UTM)
grid["county"] = gpd.sjoin(cent, cty[["county", "geometry"]], how="left", predicate="within").groupby(level=0)["county"].first()

done_area = cty[cty.county.isin(COUNTIES)]
xmin, ymin, xmax, ymax = done_area.total_bounds
maps_fig, axs = plt.subplots(1, 3, figsize=(20, 8))
from matplotlib.colors import LogNorm
panels = [("roof_m2", "Poultry-house roof, m² per 5 km cell", "Greens", None),
          ("corn_acres", f"Corn, acres per 5 km cell (CDL {YEAR})", "YlOrBr", None),
          ("index_haul", f"m² of house per corn acre within {HAUL_KM} km", "RdPu", LogNorm(vmin=0.1, vmax=50))]
for ax, (col, title, cmap, norm) in zip(axs, panels):
    cty.plot(ax=ax, color="#f4f6f4", edgecolor="#9aa59a", lw=0.5)
    sub = grid[(grid[col] > 0) & np.isfinite(grid[col])] if col != "index_haul" else grid[(grid.roof_m2 > 0) & np.isfinite(grid[col])]
    sub.plot(ax=ax, column=col, cmap=cmap, norm=norm, legend=True, legend_kwds={"shrink": 0.6})
    ax.set_xlim(xmin - 5000, xmax + 5000); ax.set_ylim(ymin - 5000, ymax + 5000)
    ax.set_title(title); ax.set_axis_off()
plt.tight_layout()

## 4. How concentrated is the litter relative to the corn?

Sort the cells that have houses from highest to lowest index (within hauling distance). For the top cells: what share of all house roof do they hold, and what share of the corn? If the litter were spread like the corn, the two curves would match.

In [ ]:
hc = grid[grid.roof_m2 > 0].sort_values("index_haul", ascending=False).copy()
hc["cum_roof"] = hc.roof_m2.cumsum() / hc.roof_m2.sum()
hc["cum_corn"] = hc.corn_acres.cumsum() / grid.corn_acres[grid.county.isin(COUNTIES)].sum()
k = int((hc.cum_roof < 0.5).sum()) + 1
print(f"Half of all poultry-house roof is in {k} of {len(hc)} cells with houses; "
      f"those cells hold {hc.cum_corn.iloc[k - 1]:.0%} of the corn in the finished counties.")
top = hc.head(15)[["cell_id", "county", "houses", "roof_m2", "corn_acres", f"roof_m2_{tag}", f"corn_acres_{tag}", "index_haul"]]
display(top.round(1))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, len(hc) + 1), hc.cum_roof, label="share of house roof", color="#2c8c3c")
ax.plot(np.arange(1, len(hc) + 1), hc.cum_corn, label="share of corn", color="#c47f00")
ax.set_xlabel(f"cells, highest m²/acre within {HAUL_KM} km first"); ax.legend(); ax.set_title("Litter vs corn, cumulative");

## 5. Does the picture hold up?

Rank correlation (Spearman) of the within-reach index over cells with houses, changing one thing at a time. Close to 1 means the same places stay on top.
- **Rotation:** corn in 2023 only vs corn averaged over 2021–2024 (fields rotate corn / soybeans / wheat, corn / hay).
- **Crops:** corn only vs corn + soybeans + small grains. Litter also goes on those crops, so this matters.
- **Hauling distance:** 10 / 25 / 50 km.

In [ ]:
wh = grid[grid.roof_m2 > 0]
def idx(r, crop):
    return pd.Series(B.roof_per_acre(wh[f"roof_m2_{r}km"], wh[f"{crop}_acres_{r}km"]), index=wh.index)
base = idx(HAUL_KM, "corn")
checks = {f"corn {ROTATION_YEARS[0]}-{ROTATION_YEARS[-1]} mean": idx(HAUL_KM, "corn_rotation_mean"),
          "corn + soybeans + small grains": idx(HAUL_KM, "grain")}
checks.update({f"{r} km instead of {HAUL_KM} km": idx(r, "corn") for r in RADII_KM if r != HAUL_KM})
fin = lambda s: s.replace([np.inf, -np.inf], np.nan)
robust = pd.DataFrame({"spearman_vs_base": {k: fin(base).corr(fin(v), method="spearman") for k, v in checks.items()},
                       "median index (base 2023 corn)": float(fin(base).median()),
                       "median index (this variant)": {k: float(fin(v).median()) for k, v in checks.items()}})
display(robust.round(3))

## 6. County cross-check: birds per m² of house

Reported inventory (AWTF 2017, Table 1a.3) divided by our detected roof area, scaled up by that county's measured recall (share of hand-traced houses we find). It's a check on the Extension stocking number, not a replacement for it. Caveats: inventory is 2017 and houses are 2023; inventory is birds on hand on one day (houses empty between flocks count as 0); it includes layers; recall exists only where USGS hand tracing does.

In [ ]:
rows = []
for n in COUNTIES:
    s = summaries[n]; t = s["totals"]; a = s["accuracy"]
    roof = by_county.roof_m2.get(n, 0.0)
    rec = a["recall"] if a else None
    for label, birds in [("AWTF Census 2017", t["AWTF 2017 inventory, USDA Ag Census (birds)"]),
                         ("AWTF MDE CAFO 2017", t["AWTF 2017 inventory, MDE CAFO data (birds)"])]:
        rows.append({"county": n, "inventory": label, "birds": birds, "roof_m2": roof, "recall": rec,
                     "birds_per_m2": B.implied_birds_per_m2(birds, roof, rec) if birds and rec else None})
stock = pd.DataFrame(rows).pivot(index="county", columns="inventory", values="birds_per_m2")
stock["roof_m2"] = by_county.roof_m2; stock["recall"] = [summaries[n]["accuracy"]["recall"] if summaries[n]["accuracy"] else None for n in stock.index]
display(stock.round(2))
w = stock.dropna(subset=["AWTF MDE CAFO 2017"])
print("roof-weighted mean, MDE CAFO inventory:",
      round(float(np.average(w["AWTF MDE CAFO 2017"], weights=w.roof_m2)), 2) if len(w) else None, "birds per m²")

## 7. What the N / P map still needs (the ask list for the Extension meeting)

Every value marked *pending* has to come from Ginny Rhodes / John Moyle / UMD nutrient recommendations (with the AWTF 2023 report). When they're filled in, set them here with their source and rerun: `B.nutrient_balance()` then gives N and P₂O₅ supply, demand and balance per cell. Until then it refuses, on purpose: there is no N or P number in this notebook. Also on the list: is one set of coefficients right for the whole Eastern Shore (bird size, flocks per year, yield goals)? And should P set the limit? In Maryland, P on high-P soils usually limits litter use.

In [ ]:
coeffs = json.loads(json.dumps(B.COEFFICIENTS))   # a copy to fill in; the template stays as is
# coeffs["birds_per_m2"]["value"] = ...; coeffs["birds_per_m2"]["source"] = "..."
display(B.coefficient_table(coeffs))
try:
    nb_grid = B.nutrient_balance(grid, coeffs)
    print("N/P balance computed (coefficients as above)")
except ValueError as e:
    nb_grid = None
    print("No N/P balance yet --", e)

## 8. Save to the shared drive

The grid is aggregated to 5 km cells and names no farm. It still goes to the shared drive with the other results, not to git.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
save = grid.copy()
for c in save.columns:
    if save[c].dtype.kind == "f":
        save[c] = save[c].replace([np.inf, -np.inf], np.nan)   # GeoJSON has no inf: "houses but no corn" -> null
save.to_crs(4326).to_file(OUT / f"balance_grid_{CELL_KM}km.geojson", driver="GeoJSON")
by_county.join(stock.drop(columns=["roof_m2"])).to_csv(OUT / "county_houses_and_stocking.csv")
robust.to_csv(OUT / "robustness.csv")
maps_fig.savefig(OUT / "maps.png", dpi=150, bbox_inches="tight")
meta = {"run_at": pd.Timestamp.now().isoformat(timespec="seconds"), "git_commit": git_commit(),
        "counties": COUNTIES, "houses": int(len(houses_gdf)), "cell_km": CELL_KM, "haul_km": HAUL_KM, "radii_km": RADII_KM,
        "cdl": {str(y): {k: v for k, v in i.items() if k != "path"} for y, i in cdl_info.items()},
        "crop_groups": cdl.CROP_GROUPS, "coefficients": coeffs, "nutrient_balance_computed": nb_grid is not None,
        "environment": env.environment()}
(OUT / "run_meta.json").write_text(json.dumps(meta, indent=2, default=str))
print("saved to", OUT)

## What this shows

*(Write from the outputs above once notebook 05 has finished the counties: how concentrated the house roof is relative to corn (section 4), which areas top the within-reach index, whether that holds across rotation / crops / distance (section 5), what birds per m² the inventories imply (section 6), and what is still needed before N and P (section 7).)*